# 03 | Foreign Key and Table Relationship Analysis

**Purpose:** Check whether key values in child tables match their referenced parent tables, and inspect geographic-prefix and product-category mapping gaps.

**Inputs:** The nine original Olist CSV files in `data/raw/`.

**Interpretation:** Unmatched values are data-quality or mapping findings to investigate; they are not automatically a reason to delete records. Geolocation zip-prefix checks are membership tests against the geolocation table, which can contain repeated prefixes.

Run after `01_dataset_audit.ipynb` and `02_primary_key_identifier_analysis.ipynb`.


In [ ]:
import pandas as pd
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = Path(os.environ.get("OLIST_DATA_DIR", PROJECT_ROOT / "data" / "raw")).expanduser()

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Olist CSV folder not found: {DATA_DIR}\n"
        "Download the Olist CSV files and place them in data/raw, "
        "or set the OLIST_DATA_DIR environment variable."
    )

print("Dataset folder:", DATA_DIR)


## Load the source tables


In [ ]:
datasets = {}

for file in DATA_DIR.glob("*.csv"):
    datasets[file.stem] = pd.read_csv(file, low_memory=False)

print("Datasets loaded:", len(datasets))

## Validate key relationships between tables


In [ ]:
relationship_results = []

for child, child_column, parent, parent_column in relationships:
    child_df = datasets[child]
    parent_values = datasets[parent][parent_column].dropna()
    missing_count = int(child_df[child_column].isna().sum())
    unmatched_mask = child_df[child_column].notna() & ~child_df[child_column].isin(parent_values)
    unmatched_count = int(unmatched_mask.sum())

    relationship_results.append({
        "Child Table": child,
        "Parent Table": parent,
        "Child Key": child_column,
        "Parent Key": parent_column,
        "Child Rows": len(child_df),
        "Missing Child Keys": missing_count,
        "Non-null Unmatched Rows": unmatched_count,
        "Distinct Unmatched Values": int(child_df.loc[unmatched_mask, child_column].nunique()),
    })

relationship_df = pd.DataFrame(relationship_results)
display(relationship_df)

## Check customer and seller zip-prefix coverage

Null or blank prefixes are counted separately from non-null prefixes that do not appear in the geolocation reference. Prefixes are normalized to five-character strings for the membership check.

In [ ]:
geo = datasets["olist_geolocation_dataset"]
geo_prefix_column = "geolocation_zip_code_prefix"

def normalize_zip_prefix(values):
    numeric_values = pd.to_numeric(values, errors="coerce").astype("Int64")
    return numeric_values.astype("string").str.zfill(5)

geo_prefixes = set(normalize_zip_prefix(geo[geo_prefix_column]).dropna())
zip_results = []

for table, column in [
    ("olist_customers_dataset", "customer_zip_code_prefix"),
    ("olist_sellers_dataset", "seller_zip_code_prefix"),
]:
    df = datasets[table]
    raw_prefixes = df[column].astype("string").str.strip()
    missing_mask = raw_prefixes.isna() | raw_prefixes.eq("")
    normalized_prefixes = normalize_zip_prefix(df[column])
    unmatched_mask = ~missing_mask & ~normalized_prefixes.isin(geo_prefixes)

    zip_results.append({
        "Dataset": table,
        "Total Rows": len(df),
        "Missing Prefix Rows": int(missing_mask.sum()),
        "Non-null Unmatched Rows": int(unmatched_mask.sum()),
        "Distinct Unmatched Prefixes": int(normalized_prefixes[unmatched_mask].nunique()),
    })

zip_coverage_df = pd.DataFrame(zip_results)
display(zip_coverage_df)

## Check product-category translation coverage

Missing product categories are counted separately from non-null categories that do not have a matching translation.

In [ ]:
products = datasets["olist_products_dataset"]
translation = datasets["product_category_name_translation"]
category_column = "product_category_name"

# Handle an optional byte-order-mark or malformed BOM prefix in the translation header.
translation_category_column = next(
    (column for column in translation.columns
     if str(column).replace("\ufeff", "").replace("ï»¿", "") == category_column),
    None,
)
if translation_category_column is None:
    raise KeyError(f"Could not find {category_column!r} in the category translation table")

translated_categories = set(
    translation[translation_category_column].dropna().astype("string").str.strip()
)
product_categories = products[category_column].astype("string").str.strip()
missing_category_mask = product_categories.isna() | product_categories.eq("")
untranslated_mask = ~missing_category_mask & ~product_categories.isin(translated_categories)

category_coverage = pd.DataFrame([{
    "Product Rows": len(products),
    "Missing Category Rows": int(missing_category_mask.sum()),
    "Non-null Untranslated Rows": int(untranslated_mask.sum()),
    "Distinct Untranslated Categories": int(product_categories[untranslated_mask].nunique()),
}])

display(category_coverage)
display(product_categories[untranslated_mask].value_counts().rename("Product Rows"))